In [4]:
import pandas as pd
import joblib
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("../data/raw_data/trees_indian_dataset.csv")

# =========================
# TARGET
# =========================
target = "Tree_Health_Status"

# =========================
# 1. FREQUENCY ENCODING (FIXED: handle unseen values)
# =========================
freq_cols = ["State_Province", "City", "Tree_Name"]

freq_maps = {}

for col in freq_cols:
    freq = df[col].value_counts() / len(df)
    
    # FIX: fill unseen values with 0
    df[col] = df[col].map(freq).fillna(0)
    
    freq_maps[col] = freq

joblib.dump(freq_maps, "../models/frequency_maps.pkl")

# =========================
# 2. SOIL TYPE (FIXED TRUE/FALSE ISSUE)
# =========================
df = pd.get_dummies(df, columns=["Soil_Type"], drop_first=True)

# FIX: convert bool → int (IMPORTANT)
soil_cols = [c for c in df.columns if "Soil_Type_" in c]
df[soil_cols] = df[soil_cols].astype(int)

# =========================
# 3. BINARY ENCODING (CLEANED)
# =========================
binary_map = {
    "Yes": 1,
    "No": 0
}

binary_cols = ["Pest_Presence", "Fungal_Infection"]

for col in binary_cols:
    df[col] = df[col].map(binary_map).fillna(0)

# =========================
# 4. ORDINAL ENCODING (FIXED SAFETY)
# =========================
ordinal_cols = [
    "Growth_Stage",
    "Leaf_Color",
    "Disease_Symptoms",
    "Bark_Damage",
    "Root_Condition",
    "Watering_Frequency",
    "Fertilizer_Usage",
    "Treatment_History",
    "Inspection_Frequency"
]

ordinal_categories = [
    ["Sapling", "Juvenile", "Mature", "Ancient"],
    ["Pale Green", "Yellowish", "Vibrant Green", "Deep Green", "Brownish-Spots"],
    ["No Disease", "Leaf Blight", "Bark Canker", "Root Rot"],
    ["No Damage", "Minor", "Moderate", "Severe"],
    ["Healthy", "Damaged", "Rotting"],
    ["Weekly", "Bi-weekly", "Daily"],
    ["No Fertilizer", "Organic Compost", "NPK Chemical"],
    ["No History", "Treated", "Under Treatment"],
    ["Weekly", "Monthly", "Quarterly"]
]

ordinal_encoder = OrdinalEncoder(categories=ordinal_categories)

df[ordinal_cols] = ordinal_encoder.fit_transform(df[ordinal_cols])

joblib.dump(ordinal_encoder, "../models/ordinal_encoder.pkl")

# =========================
# 5. LABEL ENCODING (TARGET)
# =========================
label_encoder = LabelEncoder()
df[target] = label_encoder.fit_transform(df[target])

joblib.dump(label_encoder, "../models/label_encoder.pkl")

# =========================
# FINAL OUTPUT
# =========================
print("Final dataset shape:", df.shape)
df.to_csv("../data/clean_data/encoded_tree_dataset_final.csv", index=False)

Final dataset shape: (20000, 52)
